In [ ]:
import getpass
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv


load_dotenv()

llm=ChatOpenAI(model="deepseek-chat",
               base_url=os.getenv("DEEPSEEK_BASE_URL"),
               api_key=os.getenv("DEEPSEEK_API_KEY"),
               temperature=0)

In [ ]:
print(llm.invoke("你好，测试连通性").content)

In [ ]:
import operator
from typing import Annotated

from langgraph.graph import StateGraph,MessagesState,START,END

class AgentState(MessagesState):
    next:str
    step_count:Annotated[int,operator.add]
    


In [ ]:
members=["chat","coder","sqler"]
options=members+["FINISH"]


In [ ]:
options

In [ ]:
from typing import Literal
from typing_extensions import TypedDict

class Router(TypedDict):
    """Worker to route to next. If no workers needed, route to FINISH"""
    next: Literal["chat", "coder", "sqler", "FINISH"]

In [ ]:
from langchain_core.messages import AnyMessage, SystemMessage,HumanMessage,ToolMessage
MAX_STEPS=4
def supervisor(state:AgentState):
    if state.get("step_count",0) >= MAX_STEPS:
        return {"next":"FINISH"}
    system_prompt=(
        "You are a supervisor tasked with managing a conversation between the "
        f" following workers: {members}. \n\n"
        "Each worker has a specific role: \n"
        "- chat: Responds directly to user inputs using natural language.\n"
        "- coder: Activated for tasks that require mathematical calculations or specific coding needs \n"
        "- sqler: Used when database queries or explicit SQL generation is needed.\n\n"
        "Given the following user request, respond with the worker to act next."
        " Each worker will perform a task and respond with their results and status."
        "When finished , respond with FINISH."
    )
    
    messages=[{"role":"system","content":system_prompt},]+state["messages"]
    response=llm.with_structured_output(Router,method="function_calling").invoke(messages)
    count=state.get("step_count")
    print(f"step count: {count}")
    return {"next": response['next'],"step_count":1}

In [ ]:
def chat(state:AgentState):
    messages=state["messages"][-1]
    model_response=llm.invoke(messages.content)
    final_response=[HumanMessage(content=model_response.content,name="chat")]
    return {"messages":final_response}


def coder(state:AgentState):
    messages=state["messages"][-1]
    model_response=llm.invoke(messages.content)
    final_response=[HumanMessage(content=model_response.content,name="coder")]
    return {"messages":final_response}

def sqler(state:AgentState):
    messages=state["messages"][-1]
    model_response=llm.invoke(messages.content)
    final_response=[HumanMessage(content=model_response.content,name="sqler")]
    return {"messages":final_response}

In [ ]:
builder=StateGraph(AgentState)

builder.add_node("supervisor",supervisor)
builder.add_node("chat",chat)
builder.add_node("coder",coder)
builder.add_node("sqler",sqler)

In [ ]:
for member in members:
    builder.add_edge(member,"supervisor")
    


In [ ]:
builder.add_edge(START, "supervisor")
builder.add_conditional_edges(
    "supervisor",
    lambda state: state["next"],
    {
        "chat": "chat",
        "coder": "coder",
        "sqler": "sqler",
        "FINISH": END,
    },
)

graph = builder.compile()

In [ ]:
from IPython.display import display,Image

display(Image(graph.get_graph(xray=True).draw_mermaid_png()))

In [ ]:
for chunk in graph.stream({"messages":"你好，什么是机器学习"},stream_mode="values"):
    print(chunk)

In [ ]:
for chunk in graph.stream({"messages":"你好，帮我生成一个二分查找的Python代码"},stream_mode="values"):
    print(chunk)

In [ ]:
for chunk in graph.stream({"messages":"我想查询数据库中data表的所有数据"},{"recursion_limit":20},stream_mode="values"):
    print(chunk)